# 🔧 Nexus Audio - Tokenização de Dados

**Este notebook roda UMA VEZ** para converter áudio em tokens EnCodec.

Depois de rodar, baixe os tokens e use no notebook de treino rápido!

In [ ]:
# Setup
!pip install -q torch torchaudio einops encodec

In [ ]:
import torch
import torchaudio
import os
from pathlib import Path
from tqdm.notebook import tqdm
from encodec import EncodecModel
from encodec.utils import convert_audio

print(f"CUDA: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Carregar EnCodec
encodec = EncodecModel.encodec_model_24khz()
encodec.set_target_bandwidth(6.0)
encodec = encodec.to(device)
encodec.eval()
print("✅ EnCodec carregado!")

In [ ]:
# Configuração
# FMA_PATH = "/kaggle/input/fma-small"  # Dataset original
FMA_PATH = "/kaggle/input/nexus-fma-processed"  # OU chunks já processados

OUTPUT_DIR = "/kaggle/working/tokens"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SEC = 10
SAMPLE_RATE = 24000
CHUNK_SAMPLES = CHUNK_SEC * SAMPLE_RATE

In [ ]:
def tokenize_audio(waveform, sample_rate=24000):
    """Converte waveform em tokens EnCodec."""
    # Garantir formato correto
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:  # Stereo -> Mono
        waveform = waveform.mean(dim=0, keepdim=True)
    
    # Converter para EnCodec format
    wav = convert_audio(waveform, sample_rate, 24000, 1)
    wav = wav.unsqueeze(0).to(device)  # [1, 1, samples]
    
    # Tokenizar
    with torch.no_grad():
        encoded = encodec.encode(wav)
        # encoded[0] shape: [1, n_codebooks, seq_len]
        tokens = encoded[0][0]  # [n_codebooks, seq_len]
    
    return tokens.cpu()

In [ ]:
# Verificar se são chunks .pt ou áudios .mp3/.wav
pt_files = list(Path(FMA_PATH).glob("**/*.pt"))
mp3_files = list(Path(FMA_PATH).glob("**/*.mp3"))
wav_files = list(Path(FMA_PATH).glob("**/*.wav"))

print(f"Arquivos .pt: {len(pt_files)}")
print(f"Arquivos .mp3: {len(mp3_files)}")
print(f"Arquivos .wav: {len(wav_files)}")

# Decidir qual usar
if len(pt_files) > 0:
    files = pt_files
    file_type = "pt"
elif len(mp3_files) > 0:
    files = mp3_files
    file_type = "audio"
else:
    files = wav_files
    file_type = "audio"

print(f"\nUsando {len(files)} arquivos ({file_type})")

In [ ]:
# Tokenizar tudo!
success = 0
errors = 0

for i, file_path in enumerate(tqdm(files, desc="Tokenizando")):
    try:
        # Carregar áudio
        if file_type == "pt":
            waveform = torch.load(file_path)
            sr = 24000  # Já está em 24kHz
        else:
            waveform, sr = torchaudio.load(str(file_path))
            # Resample se necessário
            if sr != 24000:
                waveform = torchaudio.transforms.Resample(sr, 24000)(waveform)
        
        # Se for áudio longo, dividir em chunks
        if waveform.shape[-1] > CHUNK_SAMPLES:
            for j in range(0, waveform.shape[-1] - CHUNK_SAMPLES, CHUNK_SAMPLES):
                chunk = waveform[..., j:j + CHUNK_SAMPLES]
                tokens = tokenize_audio(chunk)
                out_name = f"{file_path.stem}_{j}.tokens.pt"
                torch.save(tokens, f"{OUTPUT_DIR}/{out_name}")
                success += 1
        else:
            # Chunk único
            tokens = tokenize_audio(waveform)
            out_name = f"{file_path.stem}.tokens.pt"
            torch.save(tokens, f"{OUTPUT_DIR}/{out_name}")
            success += 1
            
    except Exception as e:
        errors += 1
        if errors < 5:
            print(f"Erro em {file_path.name}: {e}")

print(f"\n✅ Tokenizados: {success}")
print(f"❌ Erros: {errors}")

In [ ]:
# Verificar resultado
token_files = list(Path(OUTPUT_DIR).glob("*.pt"))
print(f"\n📦 Total de tokens salvos: {len(token_files)}")

# Ver tamanho de um token
if token_files:
    sample = torch.load(token_files[0])
    print(f"Shape de um token: {sample.shape}")
    print(f"Tipo: {sample.dtype}")

In [ ]:
# Copiar pra output
!cp -r /kaggle/working/tokens /kaggle/working/
!ls -la /kaggle/working/tokens/ | head -20
print("\n✅ Tokens salvos em /kaggle/working/tokens/")
print("\n📥 Baixe essa pasta e crie um novo Dataset no Kaggle!")